# English reconstruction with RNN and LSTM encoder–decoders
## From encoder states to an attention context, then the next word

We reconstruct the same four-word English sentences used in the self-attention exercise. There are **two separate models**, each with additive attention: a simple RNN and an LSTM. This is a sequence reconstruction task, not translation between languages.

Begin with the overview below. Then follow the dataset, token shifts, context arithmetic, model implementations, training, free-running predictions and evaluation. The notebook runs in local Jupyter or Google Colab; dependencies install if missing. The synthetic dataset requires no download.

Four source words produce **h1–h4** in this notebook. We do not append EOS to the encoder input. If an encoder also reads EOS, it would produce h5. The decoder does predict EOS after reconstructing the four words.

In [ ]:
import importlib.util, subprocess, sys
required = ['torch', 'numpy', 'pandas', 'matplotlib']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
from pathlib import Path
import random, math, time, copy, json, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from IPython.display import display, HTML
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULTS = Path('results_recurrent_reconstruction'); RESULTS.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 115, 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})
display(HTML('<style>.text_cell_render,.jp-MarkdownOutput{font-family:Arial,sans-serif;font-size:18px;line-height:1.85;max-width:1100px}.text_cell_render p{margin:18px 0}.text_cell_render h2{margin-top:40px;color:#17324d}</style>'))
print('PyTorch:', torch.__version__, '| device:', DEVICE)

## 1 · Start with the general block diagram
The encoder makes four contextual states. Its last state initializes the decoder, while **all four states remain available as attention memory**. At output step t:

1. Compare the previous decoder state s(t−1) with h1, h2, h3 and h4.
2. Turn the four scores into weights and form context c(t).
3. Use c(t), the previous word embedding and the previous decoder state to form s(t).
4. Use s(t) **and c(t)** in a fully connected output head to predict the next word.

The context is a weighted summary of source information for this output step. It changes from step to step; the four encoder states remain fixed during decoding. This attention-conditioned architecture is a teaching choice, not an exact reproduction of the original Bahdanau model.

In [ ]:
def overview_diagram():
    fig,ax=plt.subplots(figsize=(15,8));ax.set(xlim=(0,15),ylim=(0,8));ax.axis('off')
    def box(x,y,w,h,text,color='#def3f0'):
        ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=.04',fc=color,ec='#087f8c',lw=1.5))
        ax.text(x+w/2,y+h/2,text,ha='center',va='center',fontsize=11)
    def arrow(a,b,color='#087f8c'):
        ax.annotate('',xy=b,xytext=a,arrowprops=dict(arrowstyle='->',lw=1.7,color=color))
    for i,word in enumerate(['small','cats','eat','fish']):
        x=.3+3.65*i
        box(x,6.7,2.6,.8,word,'#dcecf6');arrow((x+1.3,6.7),(x+1.3,6.2))
        box(x,5.35,2.6,.8,f'Encoder state h{i+1}')
        if i<3:arrow((x+2.65,5.75),(x+3.55,5.75))
        arrow((x+1.3,5.3),(7.4,4.55))
    box(5.2,3.55,4.5,1,'Attention over h1, h2, h3, h4\nWeights → context c(t)','#fff0c7')
    box(.3,3.55,3.7,1,'Previous decoder state s(t−1)\nAt step 1: s0 = h4','#dcecf6');arrow((4,4.05),(5.15,4.05))
    box(.3,1.5,3.7,1,'Previous word embedding\nAt step 1: <BOS>','#dcecf6')
    box(5.2,1.5,4.5,1,'Decoder update → s(t)\nUses previous state, word and c(t)');arrow((4,2),(5.15,2));arrow((2.1,3.5),(5.2,2.3));arrow((7.4,3.5),(7.4,2.55))
    box(10.8,1.5,3.9,1,'Linear([s(t); c(t)])\nVocabulary scores → word','#fff0c7');arrow((9.75,2),(10.75,2))
    arrow((9.75,4.05),(12.8,4.05));arrow((12.8,4.05),(12.8,2.55))
    ax.text(7.5,.5,'Next step: use the new state and the preceding word; recompute attention and context.',ha='center',fontsize=12)
    fig.tight_layout();plt.show()
from matplotlib.patches import FancyBboxPatch
overview_diagram()

## 2 · The same 512 sentences and the same split
Use exactly the same grammar, seed and 80/10/10 split as the previous notebook. Each sentence has an adjective, animal, verb and food. Test sentences are unseen combinations, but the vocabulary and grammar are shared. This is a controlled exercise, not a general-language benchmark.

The encoder and decoder have separate embedding layers. No positional embedding is added: the recurrent order already distinguishes time steps.

In [ ]:
adjectives = ['small','young','hungry','happy','quiet','gentle','sleepy','lively',
              'playful','curious','healthy','active','little','calm','friendly','eager']
foods = {'cats':['fish','tuna'], 'dogs':['biscuits','bones'], 'cows':['grass','hay'],
         'birds':['seeds','worms'], 'rabbits':['carrots','lettuce'],
         'horses':['oats','apples'], 'pandas':['bamboo','shoots'], 'monkeys':['bananas','mangoes']}
sentences = [' '.join([adj, animal, verb, food])
             for adj in adjectives for animal, options in foods.items()
             for verb in ['eat','enjoy'] for food in options]
rng = np.random.default_rng(SEED)
order = rng.permutation(len(sentences))
a, b = int(.8*len(order)), int(.9*len(order))
train_text = [sentences[i] for i in order[:a]]
val_text = [sentences[i] for i in order[a:b]]
test_text = [sentences[i] for i in order[b:]]
assert not (set(train_text)&set(val_text) or set(train_text)&set(test_text) or set(val_text)&set(test_text))
# Learn vocabulary from training only; MASK and UNK are special tokens.
vocab = ['[UNK]', '[MASK]', '<BOS>', '<EOS>'] + sorted(set(' '.join(train_text).split()))
to_id = {word:i for i,word in enumerate(vocab)}
def encode(texts):return torch.tensor([[to_id.get(w,0) for w in s.split()] for s in texts])
def decode(ids):return ' '.join(vocab[int(i)] for i in ids)
train_y, val_y, test_y = map(encode, [train_text,val_text,test_text])
assert not (train_y.eq(0).any() or val_y.eq(0).any() or test_y.eq(0).any())
display(pd.DataFrame({'split':['train','validation','test'], 'sentences':[len(train_y),len(val_y),len(test_y)]}))
display(pd.DataFrame({'example':train_text[:8]}))
fig,ax=plt.subplots(figsize=(8,3));ax.bar(['Train','Validation','Test'],[len(train_y),len(val_y),len(test_y)],color=['#087f8c','#b68013','#7395b0'])
ax.set(ylabel='Four-word sentences',title=f'{len(sentences)} unique sentences; vocabulary size {len(vocab)}');plt.show()

## 3 · Decoder inputs are shifted targets
For source “small cats eat fish”, the encoder reads four tokens. The decoder training input is **BOS small cats eat fish** and its labels are **small cats eat fish EOS**. There are five decoder steps.

At the first step no output word has been generated: BOS is the start input and s0 is the encoder final hidden state. During teacher forcing, correct previous words are supplied. During free-running evaluation, previous predictions are supplied. The target word being predicted is never supplied as that step’s decoder input.

This fixed-length dataset needs no padding. For variable-length data, mask source padding in attention and ignore padded target positions in the loss.

In [ ]:
BOS,EOS=to_id['<BOS>'],to_id['<EOS>']
def decoder_pairs(clean):
    start=torch.full((len(clean),1),BOS,dtype=torch.long)
    end=torch.full((len(clean),1),EOS,dtype=torch.long)
    return torch.cat([start,clean],1),torch.cat([clean,end],1)
train_in,train_targets=decoder_pairs(train_y)
val_in,val_targets=decoder_pairs(val_y)
test_in,test_targets=decoder_pairs(test_y)
display(pd.DataFrame({'step':range(1,6),
    'decoder_input':[vocab[int(i)] for i in train_in[0]],
    'target':[vocab[int(i)] for i in train_targets[0]]}))
print('Source:',tuple(train_y.shape),'| decoder inputs:',tuple(train_in.shape),'| targets:',tuple(train_targets.shape))

## 4 · Additive attention: how context is formed
For decoder step t and source position i:

$$e_{t,i}=v^T\tanh(W_hh_i+W_ss_{t-1}+b),\quad
\alpha_{t,i}=\frac{\exp e_{t,i}}{\sum_j\exp e_{t,j}},\quad
c_t=\sum_i\alpha_{t,i}h_i.$$

The score network is a small shared feed-forward network. It does not update a recurrent state. W_h and W_s project the two kinds of state to a common matching space. Tanh makes their interaction nonlinear. The final linear layer v turns those features into one scalar score. Row softmax creates four mixing weights.

All h_i have 64 features, so c_t also has 64 features. One weight scales every feature of its corresponding h_i. We add the weighted vectors; we do not average the features within a vector. There is no separate attention target: reconstruction loss trains this scorer jointly with the encoder and decoder.

In [ ]:
EMBED,HIDDEN,ATTN = 64,64,64
class AdditiveAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.source=nn.Linear(HIDDEN,ATTN,bias=False)
        self.query=nn.Linear(HIDDEN,ATTN,bias=True)
        self.score=nn.Linear(ATTN,1,bias=False)
    def forward(self, memory, previous_state):
        # memory B × 4 × 64; previous_state B × 64
        combined=torch.tanh(self.source(memory)+self.query(previous_state)[:,None,:])
        scores=self.score(combined).squeeze(-1)       # B × 4
        weights=torch.softmax(scores,dim=-1)         # B × 4
        context=torch.bmm(weights[:,None,:],memory).squeeze(1) # B × 64
        return context,weights,scores
# A two-feature numerical illustration, not learned values:
toy_h=torch.tensor([[1.,0.],[0.,1.],[1.,1.],[2.,0.]])
toy_weights=torch.tensor([.1,.2,.6,.1])
toy_context=toy_weights@toy_h
print('Toy context = .1*[1,0] + .2*[0,1] + .6*[1,1] + .1*[2,0] =',toy_context.tolist())
display(pd.DataFrame({'state':['h1','h2','h3','h4'],'weight':toy_weights.numpy(),
 'feature_1_contribution':(toy_weights*toy_h[:,0]).numpy(),
 'feature_2_contribution':(toy_weights*toy_h[:,1]).numpy()}))

## 5 · RNN model: context enters the state and the word head
The encoder is a one-layer, forward nn.RNN. It returns all four states and the final state. Initialize s0=h4.

At each decoder step:
$$s_t=\tanh(W_x[\operatorname{Emb}(y_{t-1});c_t]+W_rs_{t-1}+b).$$
$$\operatorname{logits}_t=W_o[s_t;c_t]+b_o.$$

The concatenated decoder input has 64+64=128 features. The recurrent state has 64 features. The output head receives 128 features and returns vocabulary scores. Context therefore has two explicit routes to the prediction: through the state update and directly through the output head.

At t=1, use BOS and s0=h4. The first context can already use all four source positions, although early individual forward-encoder states only summarize prefixes.

In [ ]:
class RNNReconstructor(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.encoder_embedding=nn.Embedding(vocab_size,EMBED)
        self.encoder=nn.RNN(EMBED,HIDDEN,batch_first=True)
        self.decoder_embedding=nn.Embedding(vocab_size,EMBED)
        self.attention=AdditiveAttention()
        self.decoder=nn.RNNCell(EMBED+HIDDEN,HIDDEN)
        self.output_head=nn.Linear(2*HIDDEN,vocab_size)
    def encode(self,source):
        memory,final=self.encoder(self.encoder_embedding(source))
        return memory,final[0]
    def step(self,token,state,memory):
        context,weights,scores=self.attention(memory,state)
        previous=state
        decoder_input=torch.cat([self.decoder_embedding(token),context],dim=-1)
        state=self.decoder(decoder_input,state)
        logits=self.output_head(torch.cat([state,context],dim=-1))
        trace={'previous':previous,'context':context,'weights':weights,'scores':scores,'state':state,'logits':logits}
        return logits,state,trace
    def forward(self,source,decoder_inputs):
        memory,state=self.encode(source)
        outputs=[]
        for t in range(decoder_inputs.shape[1]):
            logits,state,_=self.step(decoder_inputs[:,t],state,memory)
            outputs.append(logits)
        return torch.stack(outputs,dim=1)
rnn=RNNReconstructor(len(vocab)).to(DEVICE)
print(rnn)
print('Parameters:',sum(p.numel() for p in rnn.parameters()))
print('Logits:',tuple(rnn(train_y[:2].to(DEVICE),train_in[:2].to(DEVICE)).shape))

## 6 · Training utilities: one next-word loss, all parameters updated
CrossEntropyLoss compares raw logits against token IDs over all five target steps, including EOS. Gradients update the word head, recurrent decoder, attention, encoder and embeddings together. Attention weights and hidden states have no supplied target labels.

We train each model separately below. This helper only handles batches, validation and checkpoint selection. The best validation-loss checkpoint is restored before test generation. Teacher-forced validation loss and free-running test accuracy answer different questions.

In [ ]:
criterion=nn.CrossEntropyLoss()
def sequence_loss(logits,targets):
    return criterion(logits.reshape(-1,logits.shape[-1]),targets.reshape(-1))
def train_reconstructor(model,epochs):
    loader=DataLoader(TensorDataset(train_y,train_in,train_targets),batch_size=64,shuffle=True)
    optimizer=torch.optim.Adam(model.parameters(),lr=.003)
    best=float('inf');history=[];start=time.perf_counter()
    for epoch in range(1,epochs+1):
        model.train();total=0
        for source,inputs,targets in loader:
            source,inputs,targets=source.to(DEVICE),inputs.to(DEVICE),targets.to(DEVICE)
            optimizer.zero_grad()
            loss=sequence_loss(model(source,inputs),targets)
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.0);optimizer.step()
            total+=loss.item()*len(source)
        model.eval()
        with torch.no_grad():
            val_loss=sequence_loss(model(val_y.to(DEVICE),val_in.to(DEVICE)),val_targets.to(DEVICE)).item()
        history.append({'epoch':epoch,'train_loss':total/len(train_y),'val_loss':val_loss})
        if val_loss<best:best=val_loss;best_epoch=epoch;best_state=copy.deepcopy(model.state_dict())
        if epoch==1 or epoch%10==0:print(f'Epoch {epoch:3d}: train={total/len(train_y):.4f}; validation={val_loss:.4f}')
    model.load_state_dict(best_state)
    seconds=time.perf_counter()-start
    print(f'Restored epoch {best_epoch}; elapsed {seconds:.2f} seconds')
    return pd.DataFrame(history),{'training_seconds':seconds,'best_epoch':best_epoch,'best_validation_loss':best}
def plot_history(history,title):
    fig,ax=plt.subplots(figsize=(8,3.5));ax.plot(history.epoch,history.train_loss,label='Training');ax.plot(history.epoch,history.val_loss,label='Validation')
    ax.set(xlabel='Epoch',ylabel='Teacher-forced cross-entropy',title=title);ax.legend();plt.show()

## 7 · Train the RNN independently
The encoder reads the four words sequentially. The decoder also makes five sequential state updates. Teacher forcing supplies the correct previous token during training; it does not remove the recurrent dependence between decoder states.

In [ ]:
torch.manual_seed(SEED)
rnn=RNNReconstructor(len(vocab)).to(DEVICE)
rnn_history,rnn_training=train_reconstructor(rnn,epochs=60)
plot_history(rnn_history,'RNN + additive attention')

## 8 · Generate and evaluate the RNN without teacher forcing
Begin with BOS. Feed each predicted token back into the next decoder step. Stop a sentence when EOS is predicted, with a maximum of eight steps to prevent an endless loop.

**Word-position accuracy:** correct predictions at the four expected word positions; missing words count as wrong. **Sentence exact match:** all four words are correct and EOS follows immediately, with no extra words. **EOS rate:** the model emits EOS within the generation budget. **Macro F1:** classification F1 averaged over word classes appearing in the clean test targets (the 4 word positions, excluding EOS).

These are reconstruction metrics on a controlled grammar, not general translation quality metrics. We also report teacher-forced token accuracy separately; it must not be confused with free-running generation.

In [ ]:
@torch.no_grad()
def generate(model,source,max_steps=8):
    model.eval();source=source.to(DEVICE)
    memory,state=model.encode(source)
    token=torch.full((len(source),),BOS,device=DEVICE,dtype=torch.long)
    sequences=[[] for _ in range(len(source))];ended=[False]*len(source);traces=[]
    for t in range(max_steps):
        logits,state,trace=model.step(token,state,memory)
        token=logits.argmax(-1)
        traces.append({k:v.detach().cpu() for k,v in trace.items()})
        for i,value in enumerate(token.tolist()):
            if not ended[i]:
                sequences[i].append(value)
                if value==EOS:ended[i]=True
        if all(ended):break
    return sequences,traces

def evaluate_model(model,name):
    sequences,_=generate(model,test_y)
    predicted=torch.full_like(test_y,-1)
    for i,seq in enumerate(sequences):
        words=seq[:seq.index(EOS)] if EOS in seq else seq
        for j,value in enumerate(words[:4]):predicted[i,j]=value
    exact=[seq==target.tolist() for seq,target in zip(sequences,test_targets)]
    classes=sorted(set(test_y.flatten().tolist()));f1s=[]
    for label in classes:
        tp=int(((predicted==label)&(test_y==label)).sum())
        fp=int(((predicted==label)&(test_y!=label)).sum())
        fn=int(((predicted!=label)&(test_y==label)).sum())
        f1s.append(2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.)
    with torch.no_grad():
        teacher=model(test_y.to(DEVICE),test_in.to(DEVICE)).argmax(-1).cpu()
    metrics={'model':name,'word_accuracy':float((predicted==test_y).float().mean()),
             'sentence_exact_match':float(np.mean(exact)),'eos_rate':float(np.mean([EOS in s for s in sequences])),
             'word_macro_f1':float(np.mean(f1s)),
             'teacher_forced_token_accuracy':float((teacher==test_targets).float().mean()),
             'parameters':sum(p.numel() for p in model.parameters())}
    examples=pd.DataFrame({'source':test_text,'generated':[' '.join(vocab[i] for i in s) for s in sequences],'exact':exact})
    display(pd.DataFrame([metrics]));display(examples.head(12))
    return metrics,examples,predicted
rnn_metrics,rnn_examples,rnn_predictions=evaluate_model(rnn,'RNN')

## 9 · LSTM: the same attention flow, with a separate cell memory
The LSTM has hidden state s(t) **and** internal cell memory m(t). We deliberately call the internal cell memory m, reserving c(t) for attention context.

The encoder returns h1–h4, final hidden h4 and final cell memory m4. Initialize decoder s0=h4 and m0=m4. Attention queries the previous **hidden** state s(t−1), not the cell memory. Then:

$$(s_t,m_t)=\operatorname{LSTMCell}([\operatorname{Emb}(y_{t-1});c_t],(s_{t-1},m_{t-1})).$$

The word head is unchanged: logits = Linear([s(t); c(t)]). The LSTM gates determine how its internal memory is retained and updated. Attention still performs a weighted retrieval from the encoder outputs.

In [ ]:
class LSTMReconstructor(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.encoder_embedding=nn.Embedding(vocab_size,EMBED)
        self.encoder=nn.LSTM(EMBED,HIDDEN,batch_first=True)
        self.decoder_embedding=nn.Embedding(vocab_size,EMBED)
        self.attention=AdditiveAttention()
        self.decoder=nn.LSTMCell(EMBED+HIDDEN,HIDDEN)
        self.output_head=nn.Linear(2*HIDDEN,vocab_size)
    def encode(self,source):
        memory,(hidden,cell)=self.encoder(self.encoder_embedding(source))
        return memory,(hidden[0],cell[0])
    def step(self,token,state,memory):
        previous,previous_cell=state
        context,weights,scores=self.attention(memory,previous)
        decoder_input=torch.cat([self.decoder_embedding(token),context],dim=-1)
        hidden,cell=self.decoder(decoder_input,(previous,previous_cell))
        logits=self.output_head(torch.cat([hidden,context],dim=-1))
        trace={'previous':previous,'context':context,'weights':weights,'scores':scores,
               'state':hidden,'cell':cell,'logits':logits}
        return logits,(hidden,cell),trace
    def forward(self,source,decoder_inputs):
        memory,state=self.encode(source);outputs=[]
        for t in range(decoder_inputs.shape[1]):
            logits,state,_=self.step(decoder_inputs[:,t],state,memory)
            outputs.append(logits)
        return torch.stack(outputs,dim=1)
lstm=LSTMReconstructor(len(vocab)).to(DEVICE)
print(lstm)
print('Parameters:',sum(p.numel() for p in lstm.parameters()))
print('Logits:',tuple(lstm(train_y[:2].to(DEVICE),train_in[:2].to(DEVICE)).shape))

## 10 · Train the LSTM independently
This is a fresh model with its own encoder, decoder, attention weights and optimizer. The data split, dimensions, batch size, learning rate and epoch budget match the RNN experiment. The LSTM has more parameters, so this is not a parameter-matched architecture study.

In [ ]:
torch.manual_seed(SEED)
lstm=LSTMReconstructor(len(vocab)).to(DEVICE)
lstm_history,lstm_training=train_reconstructor(lstm,epochs=60)
plot_history(lstm_history,'LSTM + additive attention')

## 11 · Evaluate the LSTM independently
Use exactly the same greedy generation and metrics as for the RNN. Test targets are used only for scoring, never as generation inputs.

In [ ]:
lstm_metrics,lstm_examples,lstm_predictions=evaluate_model(lstm,'LSTM')

## 12 · Trace the first word, then the entire generated sentence
For t=1, the decoder has BOS and s0=h4. Calculate four scores, softmax weights and c1. Calculate s1 using BOS and c1. Concatenate s1 with c1 to predict the first word.

This cell exposes the actual trained values and verifies the weighted sum. For the LSTM, it also shows its internal cell memory shape. The following table records every generation step, including EOS. Attention maps show which source vectors were mixed; they are not guaranteed word-alignment or causal explanations.

In [ ]:
trace_records={}
def inspect_model(model,name,source_text):
    source=encode([source_text])
    sequences,traces=generate(model,source)
    with torch.no_grad():memory,initial=model.encode(source.to(DEVICE))
    first=traces[0]
    manual=(first['weights'][0,:,None]*memory[0].cpu()).sum(0)
    assert torch.allclose(manual,first['context'][0],atol=1e-5)
    assert torch.allclose(first['weights'].sum(-1),torch.ones(1),atol=1e-6)
    hidden0=initial[0] if isinstance(initial,tuple) else initial
    assert torch.allclose(hidden0,memory[:,-1])
    print(name,'source:',source_text)
    display(pd.DataFrame({'source_word':source_text.split(),'score_for_first_word':first['scores'][0].numpy(),
                          'weight_for_first_word':first['weights'][0].numpy()}))
    for key in ['previous','context','state']:
        print(key,'shape',tuple(first[key].shape),'first 6 features',first[key][0,:6].numpy().round(4))
    if 'cell' in first:print('LSTM internal cell memory:',tuple(first['cell'].shape))
    logits_check=model.output_head(torch.cat([first['state'],first['context']],-1).to(DEVICE)).detach().cpu()
    assert torch.allclose(logits_check,first['logits'],atol=1e-5)
    probs=torch.softmax(first['logits'][0],-1);top=probs.topk(5)
    display(pd.DataFrame({'first_word_candidate':[vocab[int(i)] for i in top.indices],'probability':top.values.numpy()}))
    tokens=sequences[0];rows=[]
    for t,token in enumerate(tokens):
        rows.append({'step':t+1,'input_word':vocab[BOS if t==0 else tokens[t-1]],
                     'query_state':f's{t}','context':f'c{t+1}','new_state':f's{t+1}',
                     'prediction':vocab[token]})
    display(pd.DataFrame(rows))
    weights=torch.stack([traces[t]['weights'][0] for t in range(len(tokens))]).numpy()
    fig,ax=plt.subplots(figsize=(7,5));im=ax.imshow(weights,cmap='YlGnBu',vmin=0,vmax=1,aspect='auto')
    ax.set_xticks(range(4),source_text.split(),rotation=20);ax.set_yticks(range(len(tokens)),[vocab[i] for i in tokens])
    ax.set(xlabel='Source position / encoder state',ylabel='Generated word',title=name+' · learned cross-attention')
    for i in range(len(tokens)):
        for j in range(4):ax.text(j,i,f'{weights[i,j]:.2f}',ha='center',va='center',color='white' if weights[i,j]>.5 else 'black')
    fig.colorbar(im,ax=ax,label='Mixing weight');fig.tight_layout();plt.show()
    trace_records[name]={'source':source_text,'generated':[vocab[i] for i in tokens],
       'first_scores':first['scores'][0].tolist(),'first_weights':first['weights'][0].tolist(),
       'first_context':first['context'][0].tolist(),'first_state':first['state'][0].tolist(),
       'memory_first_two':memory[0,:,:2].cpu().tolist(),'weights':weights.tolist()}
inspect_model(rnn,'RNN',test_text[0])
inspect_model(lstm,'LSTM',test_text[0])
(RESULTS/'traces.json').write_text(json.dumps(trace_records,indent=2))

## 13 · Compare the measured outcomes
Sentence exact match includes correct termination. Word-position accuracy and macro F1 exclude EOS; teacher-forced token accuracy includes EOS. A model can have good teacher-forced performance yet accumulate errors during generation, so free-running metrics are essential.

This short copying task may be easy for both networks. It cannot establish which architecture is superior on long sequences or natural translation.

In [ ]:
comparison=pd.DataFrame([{**rnn_metrics,**rnn_training},{**lstm_metrics,**lstm_training}])
display(comparison)
fig,ax=plt.subplots(figsize=(9,4));x=np.arange(2);width=.25
for j,(key,label) in enumerate([('word_accuracy','Word accuracy'),('sentence_exact_match','Exact sentence'),('word_macro_f1','Word macro F1')]):
    ax.bar(x+(j-1)*width,comparison[key]*100,width,label=label)
ax.set_xticks(x,comparison.model);ax.set(ylim=(0,110),ylabel='Percent',title='Free-running test reconstruction');ax.legend();plt.show()
comparison.to_csv(RESULTS/'comparison.csv',index=False)
(RESULTS/'metrics.json').write_text(json.dumps({'data':{'train':len(train_y),'validation':len(val_y),'test':len(test_y),'vocabulary':len(vocab)},'models':comparison.to_dict(orient='records')},indent=2))
for name,model,history,examples in [('rnn',rnn,rnn_history,rnn_examples),('lstm',lstm,lstm_history,lstm_examples)]:
    history.to_csv(RESULTS/f'{name}_history.csv',index=False);examples.to_csv(RESULTS/f'{name}_predictions.csv',index=False)
    torch.save({'state_dict':model.state_dict(),'vocab':vocab,'embed':EMBED,'hidden':HIDDEN,'attention':ATTN},RESULTS/f'{name}_model.pt')

## 14 · What changes when we move to a Transformer?
In these models, a decoder state asks for a context from encoder states, and recurrence updates the next decoder state. Encoder self-attention instead lets every source position query all source positions within a layer, without recurrence. A full Transformer decoder adds causal target self-attention and source cross-attention.

A forward RNN’s early states lack later source words. A bidirectional encoder can remedy that; a decoder is not the only way to obtain both-sided context. A separate sequence decoder is useful for producing an output sequence with potentially different order and length. For same-position reconstruction, an encoder plus output head can also suffice, as the previous exercise showed.

**References:** [Bahdanau et al.: Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/abs/1409.0473); [PyTorch RNN](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html); [PyTorch LSTM](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html). Our forward encoder, initialization and output head are explicit teaching choices.